# RoboChallenge pi0.5 复现与提交操作手册

这个 Notebook 是本项目的核心操作入口。目标是把 Linux 机器上已有的 RoboChallenge pi0.5 多任务 baseline、样例数据、checkpoint 和官方提交入口串成可复现流程。

执行原则：

- 全部说明使用中文，代码和 API 名称保留英文。
- 默认只做轻量检查，不自动下载 TB 级数据。
- 不在 Notebook 中写入明文 `user_token`；提交时从环境变量读取。
- 先跑 mock，再跑官方 `demo.py`。
- 如果遇到编码问题，所有文本文件按 UTF-8 处理。

## 0. 路径与开关

下面几个路径是当前 Linux 机器上的真实路径。`BASELINE_DIR` 是已有的 pi0.5 多任务推理样例，`WORK_REPO` 是本仓库。

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import textwrap
import time

WORK_REPO = Path('/home/yjl/robochallenge/repo')
BASELINE_DIR = Path('/home/yjl/yjl/RoboChallenge/baseline_pi05_multitask')
OPENPI_SRC = BASELINE_DIR / 'openpi/src'
OPENPI_CLIENT_SRC = BASELINE_DIR / 'openpi/packages/openpi-client/src'
LEROBOT_SRC = Path('/home/yjl/yjl/RoboChallenge/third_party/lerobot')
BASELINE_PYTHON = BASELINE_DIR / '.venv/bin/python3'
CHECKPOINT_ROOT = Path('/home/yjl/yjl/RoboChallenge/checkpoints')
RUN_DIR = WORK_REPO / 'runs'
RUN_DIR.mkdir(parents=True, exist_ok=True)

# 默认选择已有 checkpoint 的 ALOHA baseline。需要换平台时改这里。
ROBOT_TYPE = 'aloha'  # 可选：aloha / dosw / arx5 / ur5

# 安全开关：默认不启动长时间服务、不加载 12GB checkpoint、不连接官方评测。
WRITE_MOCK_SETTINGS = True
START_MOCK_SERVER = False
RUN_POLICY_SMOKE = False
BUILD_OFFICIAL_DEMO_COMMAND = True

print('WORK_REPO =', WORK_REPO)
print('BASELINE_DIR =', BASELINE_DIR)
print('OPENPI_SRC =', OPENPI_SRC)
print('OPENPI_CLIENT_SRC =', OPENPI_CLIENT_SRC)
print('LEROBOT_SRC =', LEROBOT_SRC)
print('BASELINE_PYTHON =', BASELINE_PYTHON)
print('ROBOT_TYPE =', ROBOT_TYPE)

## 1. 通用命令执行函数

所有 shell 命令都通过这个函数执行，便于记录输出和排查失败。

In [ ]:
def run(cmd, cwd=BASELINE_DIR, timeout=120, check=True, env=None):
    if isinstance(cmd, (list, tuple)):
        printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    else:
        printable = cmd
    print(f'$ cd {cwd} && {printable}')
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        env=env,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f'命令失败，退出码={result.returncode}: {printable}')
    return result

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    print('已写入', path)

## 2. Linux、GPU、openpi 环境检查

这一节只检查环境，不修改系统。目标是确认当前机器是否能跑 pi0.5。

In [ ]:
assert BASELINE_DIR.exists(), f'找不到 baseline 目录: {BASELINE_DIR}'
assert BASELINE_PYTHON.exists(), f'找不到 baseline Python: {BASELINE_PYTHON}'

run('hostname && uname -a && nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', cwd=Path('/home/yjl'))

probe = r"""
import importlib.util, sys
print('python =', sys.executable)
for name in ['openpi', 'openpi_rtc', 'jax', 'torch', 'cv2', 'jupyterlab', 'ipykernel', 'nbformat']:
    print(f'{name}:', bool(importlib.util.find_spec(name)))
"""
run([str(BASELINE_PYTHON), '-c', probe])

## 3. 任务、机器人和 checkpoint 配置

RoboChallenge baseline 的四类机器人入口不同。当前机器已经有 ALOHA checkpoint，其他平台如果没有 checkpoint，就先不要跑推理，只做数据/配置检查。

In [ ]:
TASKS = {
    'aloha': {
        'robot_tag': 'aloha',
        'record_data_dir': '../20260413/aloha/pack_the_toothbrush_holder',
        'prompt': 'Put the toothbrush and toothpaste into the toiletries case in sequence, close the case, and then place it into the basket.',
        'action_type': 'joint',
        'image_size': '640x480',
        'checkpoint': CHECKPOINT_ROOT / 'table30v2_multitask_baseline_aloha',
        'config_name': 'cvpr_multitask_aloha_rtc',
    },
    'dosw': {
        'robot_tag': 'w1',
        'record_data_dir': '../20260413/w1/sweep_the_trash',
        'prompt': 'Sweep the trash on the table into the dustpan.',
        'action_type': 'joint',
        'image_size': '640x480',
        'checkpoint': CHECKPOINT_ROOT / 'table30v2_multitask_baseline_w1',
        'config_name': 'cvpr_multitask_dosw1_rtc',
    },
    'arx5': {
        'robot_tag': 'arx5',
        'record_data_dir': '../20260413/arx5/hang_the_cup',
        'prompt': 'Hang the cup on the rack.',
        'action_type': 'leftjoint',
        'image_size': '1280x720',
        'checkpoint': CHECKPOINT_ROOT / 'table30v2_multitask_baseline_arx5',
        'config_name': 'cvpr_multitask_arx5_rtc',
    },
    'ur5': {
        'robot_tag': 'ur5',
        'record_data_dir': '../20260413/ur5/arrange_fruits',
        'prompt': 'Arrange the fruit in the basket.',
        'action_type': 'leftjoint',
        'image_size': '640x480',
        'checkpoint': CHECKPOINT_ROOT / 'table30v2_multitask_baseline_ur5',
        'config_name': 'cvpr_multitask_ur5_rtc',
    },
}

cfg = TASKS[ROBOT_TYPE]
checkpoint = Path(cfg['checkpoint'])
summary = {
    'robot_type': ROBOT_TYPE,
    'robot_tag': cfg['robot_tag'],
    'record_data_dir': cfg['record_data_dir'],
    'checkpoint': str(checkpoint),
    'checkpoint_exists': checkpoint.exists(),
    'prompt': cfg['prompt'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
write_json(RUN_DIR / 'selected_robot_config.json', summary)

## 4. 样例数据体检

先检查 mock 数据是否存在，状态帧数和视频帧数是否一致。这个检查不会启动模型。

In [ ]:
audit_code = r"""
import json
from pathlib import Path
import cv2

baseline = Path('__BASELINE_DIR__')
# RECORD_DATA_DIR 是 mock_server/mock_settings.py 里的相对路径，必须相对 mock_server 目录解析。
record = (baseline / 'mock_server' / '__RECORD_DATA_DIR__').resolve()
result = {'record_dir': str(record), 'exists': record.exists(), 'states': {}, 'videos': {}}
if record.exists():
    for name in ['states.jsonl', 'left_states.jsonl', 'right_states.jsonl']:
        p = record / 'states' / name
        if p.exists():
            result['states'][name] = sum(1 for _ in p.open('r', encoding='utf-8'))
    for p in sorted((record / 'videos').glob('*.mp4')):
        cap = cv2.VideoCapture(str(p))
        result['videos'][p.name] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
print(json.dumps(result, ensure_ascii=False, indent=2))
""".replace('__BASELINE_DIR__', str(BASELINE_DIR)).replace('__RECORD_DATA_DIR__', cfg['record_data_dir'])

result = run([str(BASELINE_PYTHON), '-c', audit_code])
(RUN_DIR / f'data_audit_{ROBOT_TYPE}.json').write_text(result.stdout, encoding='utf-8')

## 5. 写入 mock_server 配置

这一节会把 `mock_server/mock_settings.py` 切到当前选择的机器人和样例任务。只改 baseline 运行目录，不改 GitHub 仓库里的示例文件。

In [ ]:
if WRITE_MOCK_SETTINGS:
    settings_path = BASELINE_DIR / 'mock_server/mock_settings.py'
    text = f"""# -*- coding: utf-8 -*-
REALSENSE_DEVICE_IDS = None

ROBOT_TAG = {cfg['robot_tag']!r}
RECORD_DATA_DIR = {cfg['record_data_dir']!r}
"""
    settings_path.write_text(text, encoding='utf-8')
    print(settings_path.read_text(encoding='utf-8'))
else:
    print('跳过写入 mock_server 配置')

## 6. 启动 mock server

默认不启动。需要本地模拟推理时，把 `START_MOCK_SERVER = True` 后执行这一节。服务日志写入 `runs/mock_server_<robot>.log`。

In [ ]:
if START_MOCK_SERVER:
    log_path = RUN_DIR / f'mock_server_{ROBOT_TYPE}.log'
    pid_path = RUN_DIR / f'mock_server_{ROBOT_TYPE}.pid'
    log_f = log_path.open('w', encoding='utf-8')
    proc = subprocess.Popen(
        [str(BASELINE_PYTHON), 'mock_robot_server.py'],
        cwd=str(BASELINE_DIR / 'mock_server'),
        stdout=log_f,
        stderr=subprocess.STDOUT,
        text=True,
    )
    pid_path.write_text(str(proc.pid), encoding='utf-8')
    time.sleep(3)
    print('mock server pid =', proc.pid)
    print('日志 =', log_path)
else:
    print('未启动 mock server。需要启动时，将 START_MOCK_SERVER 改为 True。')

## 7. 本地 mock 推理 smoke test

默认不执行，因为会加载大 checkpoint 并占用 GPU。确认 checkpoint 存在、mock server 已启动后，把 `RUN_POLICY_SMOKE = True` 再执行。

In [ ]:
if RUN_POLICY_SMOKE:
    assert checkpoint.exists(), f'checkpoint 不存在: {checkpoint}'
    cmd = [
        str(BASELINE_PYTHON), 'test.py',
        '--checkpoint', str(checkpoint),
        '--prompt', cfg['prompt'],
        '--robot_type', ROBOT_TYPE,
        '--action_type', cfg['action_type'],
        '--image_size', cfg['image_size'],
        '--valid_action_num', '16',
        '--duration', '0.033',
    ]
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(OPENPI_SRC), str(OPENPI_CLIENT_SRC), str(LEROBOT_SRC), env.get('PYTHONPATH', '')])
    run(cmd, cwd=BASELINE_DIR, timeout=600, env=env)
else:
    print('未执行推理 smoke test。需要执行时，将 RUN_POLICY_SMOKE 改为 True。')

## 8. 官方提交命令生成

这一节只生成命令，不直接提交。真实提交需要先在 RoboChallenge 网站创建 submission，并在 Linux 环境变量里设置：

```bash
export ROBOCHALLENGE_USER_TOKEN='你的 token'
export ROBOCHALLENGE_SUBMISSION_ID='网站详情页里的 submission id'
```

Notebook 不会打印 token 明文。

In [ ]:
if BUILD_OFFICIAL_DEMO_COMMAND:
    user_token = os.environ.get('ROBOCHALLENGE_USER_TOKEN')
    submission_id = os.environ.get('ROBOCHALLENGE_SUBMISSION_ID')
    command = [
        str(BASELINE_PYTHON), 'demo.py',
        '--user_token', '<ROBOCHALLENGE_USER_TOKEN>',
        '--submission_id', submission_id or '<ROBOCHALLENGE_SUBMISSION_ID>',
        '--checkpoint', str(checkpoint),
        '--prompt', cfg['prompt'],
        '--robot_type', ROBOT_TYPE,
        '--action_type', cfg['action_type'],
        '--image_size', cfg['image_size'],
        '--valid_action_num', '30',
        '--duration', '0.033',
    ]
    print('提交前检查：')
    print('- token 是否存在:', bool(user_token))
    print('- submission_id 是否存在:', bool(submission_id))
    print('- checkpoint 是否存在:', checkpoint.exists())
    print('\n命令模板：')
    print(' '.join(shlex.quote(x) for x in command))
else:
    print('跳过官方提交命令生成')

## 9. 交付状态记录

把本次 Notebook 运行看到的关键状态写入 `runs/notebook_preflight_status.json`，供后续自动化迭代读取。

In [ ]:
status = {
    'robot_type': ROBOT_TYPE,
    'baseline_dir': str(BASELINE_DIR),
    'baseline_python_exists': BASELINE_PYTHON.exists(),
    'checkpoint': str(checkpoint),
    'checkpoint_exists': checkpoint.exists(),
    'mock_settings_written': bool(WRITE_MOCK_SETTINGS),
    'mock_server_started': bool(START_MOCK_SERVER),
    'policy_smoke_requested': bool(RUN_POLICY_SMOKE),
    'has_user_token_env': bool(os.environ.get('ROBOCHALLENGE_USER_TOKEN')),
    'has_submission_id_env': bool(os.environ.get('ROBOCHALLENGE_SUBMISSION_ID')),
    'next_p0': '定位或搭建 openpi_rtc 训练 dry-run 入口，验证 1-step loss 前向/反向。',
}
write_json(RUN_DIR / 'notebook_preflight_status.json', status)
status

## 10. pi0.5 基模复现

`pi05_base` 是 OpenPI 的 Fine-Tuning 基模，不是 RoboChallenge 可直接提交的 policy。本节用于核对、下载、校验公开 GCS checkpoint，并可选运行 OpenPI 参数读取 smoke。


In [ ]:
RUN_PI05_BASE_PROBE = False  # 需要重新探测时改为 True
DOWNLOAD_PI05_BASE = False  # 约 11.6 GiB，确认要下载时再改为 True
LOAD_PI05_PARAMS = False    # 读取参数很重，确认要跑 smoke 时再改为 True

if RUN_PI05_BASE_PROBE:
    env = os.environ.copy()
    env['DOWNLOAD_PI05_BASE'] = '1' if DOWNLOAD_PI05_BASE else '0'
    env['LOAD_PI05_PARAMS'] = '1' if LOAD_PI05_PARAMS else '0'
    result = run(['bash', str(WORK_REPO / 'scripts/probe_pi05_base_model.sh')], cwd=WORK_REPO, timeout=3600, check=False, env=env)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise SystemExit(result.returncode)
    display(json.loads((WORK_REPO / 'runs/pi05_base_probe_status.json').read_text(encoding='utf-8')))


## 11. pi0.6 / pi0.7 公开可复现性审计

用户提醒需要补查 `pi0.6`、`pi0.7`。本节只审计当前公开 OpenPI 代码和 checkpoint 是否已有可下载 checkpoint；如果没有公开权重，不能假装已复现。


In [ ]:
RUN_PI06_PI07_AUDIT = True

if RUN_PI06_PI07_AUDIT:
    result = run(['python3', str(WORK_REPO / 'scripts/audit_pi06_pi07_public_release.py')], cwd=WORK_REPO, timeout=180, check=False)
    print(result.stdout[-4000:])
    if result.returncode not in (0, 1):
        print(result.stderr[-4000:])
        raise SystemExit(result.returncode)
    audit = json.loads((WORK_REPO / 'runs/pi06_pi07_public_audit.json').read_text(encoding='utf-8'))
    print('public_checkpoint_found =', audit.get('public_checkpoint_found'))
    print('report:', WORK_REPO / 'reports/pi06_pi07_public_release_audit.md')


## 12. Table30v2 ALOHA 最小分片映射

本节审计 `pack_the_toothbrush_holder` ALOHA 分片是否足够进入 dry-run converter：视频、状态字段、norm stats、动作维度和 OpenPI 配置是否一致。


In [ ]:
RUN_TABLE30V2_ALOHA_MAPPING_AUDIT = True

if RUN_TABLE30V2_ALOHA_MAPPING_AUDIT:
    result = run(['python3', str(WORK_REPO / 'scripts/audit_table30v2_aloha_mapping.py')], cwd=WORK_REPO, timeout=180, check=False)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise SystemExit(result.returncode)
    mapping = json.loads((WORK_REPO / 'runs/table30v2_aloha_mapping_audit.json').read_text(encoding='utf-8'))
    print('ready_for_dry_run_converter =', mapping.get('ready_for_dry_run_converter'))
    print('report:', WORK_REPO / 'reports/table30v2_aloha_mapping.md')


## 13. Table30v2 ALOHA dry-run converter

这一节只做最小可验证转换：抽样 5 帧、构造 50 步 action window，验证 `LeRobotW1DualDataConfig` 的 repack、ALOHA transform、delta action 和 pi0.5 32D padding。不会写全量 LeRobot 数据，也不会复制视频帧。


In [ ]:
RUN_TABLE30V2_ALOHA_DRY_RUN = True

if RUN_TABLE30V2_ALOHA_DRY_RUN:
    result = run(['python3', str(WORK_REPO / 'scripts/dry_run_table30v2_aloha_converter.py')], cwd=WORK_REPO, timeout=180, check=False)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError('Table30v2 ALOHA dry-run converter 执行失败')

    dry_run_status_path = WORK_REPO / 'runs/table30v2_aloha_dry_run_status.json'
    dry_run_status = json.loads(dry_run_status_path.read_text(encoding='utf-8'))
    smoke = dry_run_status['transform_smoke']
    print('passed =', dry_run_status.get('passed'))
    print('raw state/action =', smoke['raw_state_sequence_shape'], smoke['raw_action_sequence_shape'])
    print('after data transforms =', smoke['state_shape_after_data_transforms'], smoke['actions_shape_after_data_transforms'])
    print('after padding =', smoke['state_shape_after_padding'], smoke['actions_shape_after_padding'])
    print('report =', WORK_REPO / 'reports/table30v2_aloha_dry_run_converter.md')


## 14. Table30v2 ALOHA 可控分片 LeRobot writer

这一节写出本地 LeRobot repo，并用 `cvpr_multitask_aloha_rtc` 的 OpenPI dataloader 读取一批，验证 state/action/image/token 都能进入训练输入。默认写 64 帧；随后再用 CLI 写一个 `start_index=10`、`frame_count=80` 的独立分片，证明 writer 不再固定在单一短 episode。输出 repo 在 Linux 的 Hugging Face LeRobot cache 下，不提交全量数据到 Git。


In [ ]:
RUN_TABLE30V2_ALOHA_SHORT_LEROBOT = True
RUN_TABLE30V2_ALOHA_SHORT_LEROBOT_CLI = True

if RUN_TABLE30V2_ALOHA_SHORT_LEROBOT:
    result = run(['python3', str(WORK_REPO / 'scripts/write_table30v2_aloha_short_lerobot.py')], cwd=WORK_REPO, timeout=300, check=False)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError('Table30v2 ALOHA 默认短分片 writer 或 dataloader smoke 执行失败')

    short_status_path = WORK_REPO / 'runs/table30v2_aloha_short_lerobot_status.json'
    short_status = json.loads(short_status_path.read_text(encoding='utf-8'))
    smoke = short_status['dataloader_smoke']
    print('default passed =', short_status.get('passed'))
    print('default repo =', short_status['dataset']['repo_root'])
    print('default state/actions =', smoke['state_shape'], smoke['actions_shape'])

if RUN_TABLE30V2_ALOHA_SHORT_LEROBOT_CLI:
    cli_status_path = WORK_REPO / 'runs/table30v2_aloha_short_lerobot_cli_status.json'
    cli_report_path = WORK_REPO / 'reports/table30v2_aloha_short_lerobot_cli.md'
    result = run([
        'python3', str(WORK_REPO / 'scripts/write_table30v2_aloha_short_lerobot.py'),
        '--repo-id', 'robochallenge_table30v2_aloha_short_offset10',
        '--start-index', '10',
        '--frame-count', '80',
        '--overwrite',
        '--status-path', str(cli_status_path),
        '--report-path', str(cli_report_path),
    ], cwd=WORK_REPO, timeout=300, check=False)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError('Table30v2 ALOHA CLI 分片 writer 或 dataloader smoke 执行失败')

    cli_status = json.loads(cli_status_path.read_text(encoding='utf-8'))
    cli_smoke = cli_status['dataloader_smoke']
    print('cli passed =', cli_status.get('passed'))
    print('cli repo =', cli_status['dataset']['repo_root'])
    print('cli state/actions =', cli_smoke['state_shape'], cli_smoke['actions_shape'])
    print('reports =', WORK_REPO / 'reports/table30v2_aloha_short_lerobot.md', cli_report_path)


## 15. openpi_rtc 训练入口 shape smoke

这一节审计训练入口，避免把标准 `openpi/scripts/train.py` 误当作 `openpi_rtc` 训练脚本。默认运行轻量 shape smoke：读取本地短分片，构造 `cvpr_multitask_aloha_rtc` batch，并用 `jax.eval_shape` 验证 `train_step` 的前向 loss 和反向梯度图能闭合。它不加载 `pi05_base` 大权重，也不写真实训练 checkpoint。


In [ ]:
RUN_OPENPI_RTC_TRAIN_ENTRY_AUDIT = True

if RUN_OPENPI_RTC_TRAIN_ENTRY_AUDIT:
    result = run(['python3', str(WORK_REPO / 'scripts/audit_openpi_rtc_train_entry.py')], cwd=WORK_REPO, timeout=300, check=False)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError('openpi_rtc 训练入口 shape smoke 执行失败')

    train_entry_path = WORK_REPO / 'runs/openpi_rtc_train_entry_audit.json'
    train_entry = json.loads(train_entry_path.read_text(encoding='utf-8'))
    data = train_entry['dataloader_preflight']
    shape = train_entry['train_step_shape_smoke']
    source = train_entry['source_audit']
    print('passed =', train_entry.get('passed'))
    print('rtc train entry exists =', source['rtc_train_entry_exists'])
    print('standard train uses openpi_rtc =', source['standard_train_uses_openpi_rtc_training'])
    print('state/actions =', data['state_shape'], data['actions_shape'])
    print('loss/grad shapes =', shape['info_shape']['loss']['shape'], shape['info_shape']['grad_norm']['shape'])
    print('report =', WORK_REPO / 'reports/openpi_rtc_train_entry_audit.md')
